# 01 - Model Deployment Basics: Saving & Loading Models (pickle / joblib)
**Week 15-16 | Learning goals**
- Why we save trained models
- Save/load with `pickle` and `joblib`
- Save the whole preprocessing + model **Pipeline**
- Compare file size / speed, and know the safety caveats

In [1]:
import time, os, pickle, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

os.makedirs('../models', exist_ok=True)
df = pd.read_csv('../data/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## 1. Why save a model?
Training can take minutes to days. Deployment means *train once, reuse many times* (an API, an app, a batch job).
Saving = **serialization**: turning the Python object (learned weights + settings) into bytes on disk.

In [2]:
model = Pipeline([('scaler', StandardScaler()),
                  ('clf', RandomForestClassifier(n_estimators=200, random_state=42))])
model.fit(X_train, y_train)
print('Test accuracy:', accuracy_score(y_test, model.predict(X_test)))

Test accuracy: 0.956140350877193


## 2. Saving with `pickle` (Python standard library)

In [3]:
with open('../models/model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../models/model.pkl', 'rb') as f:
    loaded_pickle = pickle.load(f)
print('pickle reload accuracy:', accuracy_score(y_test, loaded_pickle.predict(X_test)))

pickle reload accuracy: 0.956140350877193


## 3. Saving with `joblib` (better for big NumPy arrays)
`joblib` is optimized for objects holding large NumPy arrays (most scikit-learn models) and supports compression.

In [4]:
joblib.dump(model, '../models/model.joblib')
joblib.dump(model, '../models/model_compressed.joblib', compress=3)

loaded_joblib = joblib.load('../models/model.joblib')
print('joblib reload accuracy:', accuracy_score(y_test, loaded_joblib.predict(X_test)))

joblib reload accuracy: 0.956140350877193


## 4. Compare size and speed

In [5]:
rows = []
for name in ['model.pkl', 'model.joblib', 'model_compressed.joblib']:
    p = f'../models/{name}'
    t = time.perf_counter()
    obj = pickle.load(open(p,'rb')) if name.endswith('.pkl') else joblib.load(p)
    rows.append({'file': name, 'size_KB': round(os.path.getsize(p)/1024,1),
                 'load_time_s': round(time.perf_counter()-t, 4)})
pd.DataFrame(rows)

,file,size_KB,load_time_s
0,model.pkl,621.3,0.0059
1,model.joblib,637.2,0.1043
2,model_compressed.joblib,127.0,0.0392


## 5. Using the saved model like a deployed service
A tiny `predict` function: load once, predict many times. Because the **scaler is inside the pipeline**, raw input is fine.

In [6]:
MODEL = joblib.load('../models/model.joblib')   # load once at startup

def predict(features: dict):
    row = pd.DataFrame([features])[X.columns]    # keep column order
    prob = MODEL.predict_proba(row)[0, 1]
    return {'prediction': int(prob >= 0.5), 'probability_benign': round(float(prob), 4)}

predict(X_test.iloc[0].to_dict())

{'prediction': 0, 'probability_benign': 0.0}

## 6. Important cautions
| Topic | Rule |
|---|---|
| **Security** | Never `pickle.load` / `joblib.load` a file from an untrusted source - loading can execute arbitrary code. |
| **Versions** | Load with the *same* scikit-learn / Python version used to save. Record versions. |
| **Pipelines** | Save preprocessing together with the model to avoid train/serve mismatch. |
| **Alternatives** | ONNX, `skops`, TorchScript / `model.save()` for cross-language or safer formats. |

In [7]:
import sklearn, sys
print('Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__, '| joblib', joblib.__version__)

Python 3.12.3 | scikit-learn 1.8.0 | joblib 1.5.3


## 7. Practice tasks
1. Train a `LogisticRegression` pipeline, save it with joblib, reload it, and confirm identical predictions.
2. Save the model with 3 different `compress` levels and plot size vs. load time.
3. Write `predict()` so it validates that all 30 features are present and raises a clear error otherwise.